In [9]:
from lib import get_model, config

model = get_model()
print(config)

{'model_provider': 'ollama', 'model': 'coolsoon/kanana-1.5-8b', 'temperature': 0}


In [10]:
from langchain_core.messages import HumanMessage

model.invoke([("user", "안녕")])

AIMessage(content='안녕하세요! 😊  \n무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'model': 'coolsoon/kanana-1.5-8b', 'created_at': '2026-09-21T06:08:30.9940864Z', 'done': True, 'done_reason': 'stop', 'total_duration': 502072300, 'load_duration': 2055500, 'prompt_eval_count': 17, 'prompt_eval_duration': 244254000, 'eval_count': 16, 'eval_duration': 253973000, 'logprobs': None, 'model_name': 'coolsoon/kanana-1.5-8b', 'model_provider': 'ollama'}, id='lc_run--01a0c294-8d1b-7043-90c1-517cc61b9470-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 16, 'total_tokens': 33})

In [19]:
# RunnableLambda
from langchain_core.runnables import RunnableLambda, chain

@chain
def func_a(message: str) -> str:
    return f"func_a:{message}"

def func_b(message: str) -> str:
    return f"func_b:{message}"

#func_a_runnable = RunnableLambda(func_a)
#func_b_runnable = RunnableLambda(func_b)

In [20]:
chain2 = func_a_runnable | func_b_runnable

chain2.invoke("안녕하세요!")

'func_b:func_a:안녕하세요!'

In [22]:
import re
from langchain_core.messages import HumanMessage
from langchain_core.runnables import chain
from langchain_core.output_parsers import StrOutputParser
from lib import get_model

@chain
def protected_message(message: str) -> list[HumanMessage]:
    pattern = re.compile(r"(010)\D*(\d{4})\D*(\d{4})")

    message = pattern.sub(r"\g<1>-****-****", message)

    return [HumanMessage(message)]

model = get_model()
parser = StrOutputParser()

chain2 = protected_message | model | parser

In [23]:
chain2.invoke("나의 전화번호는 010-1000-1000")

'죄송합니다. 전화번호를 입력해 주시면, 개인정보 보호 원칙에 따라 직접적인 전화번호 변환이나 분석은 도와드릴 수 없습니다.\n\n만약 전화번호의 특정 부분(예: 뒷자리 변경, 패턴 분석 등)이나 전화번호와 관련된 코드, 예시, 또는 전화번호 포맷 변환(예: 010-****-**** → 010-3456-7890)이 필요하시다면, 구체적으로 원하는 작업을 말씀해 주세요.  \n예를 들어,  \n- 전화번호 뒷자리 숨기기  \n- 전화번호 포맷 변경  \n- 전화번호 관련 파이썬 코드  \n등을 요청하실 수 있습니다.\n\n원하시는 작업을 알려주시면 최대한 도움드리겠습니다!'

In [25]:
for chunk in chain2.stream("나의 전화번호는 010-6856-7599입니다. 당신의 번호는?"):
    print(chunk, end="")

죄송합니다. 저는 실제로 전화번호를 저장하거나 받을 수 없습니다.  
만약 전화번호를 안전하게 보관하거나 복구를 위해 필요하다면, 공식적인 방법(휴대폰 설정, 백업 기능 등)을 이용해 주세요.

혹시 전화번호 형식에 대해 궁금한 점이나, 휴대폰에서 번호 관리하는 방법이 필요하시면 안내해 드릴 수 있습니다!

In [26]:
messages = [
    "나의  전화번호는 010-6856-7599입니다. 당신의 번호는?",
    "안녕하세요!",
    "반갑습니다."
]

for i, res in enumerate(chain2.batch(messages)):
    print(i, res)

0 죄송합니다. 저는 실제로 전화번호를 저장하거나 받을 수 없습니다.  
만약 전화번호를 안전하게 보관하거나 복구를 위해 필요하다면, 공식적인 방법(휴대폰 설정, 백업 기능 등)을 이용해 주세요.

혹시 다른 도움이 필요하신가요?
1 안녕하세요! 😊  
무엇을 도와드릴까요?
2 반갑습니다! 무엇을 도와드릴까요? 😊


In [29]:
async def async_messages():
    async for chunk in chain2.astream("나의  전화번호는 010-6856-7599입니다. 당신의 번호는?"):
        print(chunk, end="")

In [30]:
await async_messages()

죄송합니다. 저는 실제로 전화번호를 저장하거나 받을 수 없습니다.  
만약 전화번호를 안전하게 보관하거나 복구를 위해 필요하다면, 공식적인 방법(휴대폰 설정, 백업 기능 등)을 이용해 주세요.

혹시 다른 도움이 필요하신가요?